# Process1_Extract_texts.py

In [ ]:
import os
import shutil
import pandas as pd
from datetime import datetime

# 原始文件夹路径
source_dir = r"C:\Users\numinose\Desktop\Trump in factbase\All"

# 目标基础路径
base_target_dir = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\csv_raw"

# 定义时间段（Session）及其范围
sessions = {
    "First Session": (datetime(2015, 6, 16), datetime(2016, 11, 7)),
    "Second Session": (datetime(2019, 2, 11), datetime(2020, 11, 2)),
    "Third Session": (datetime(2023, 1, 28), datetime(2024, 11, 4)),
}

# 创建目标 Session 文件夹（带时间段）
session_paths = {}
for session_name, (start, end) in sessions.items():
    folder_name = f"{session_name} ({start.date()} to {end.date()})"
    full_path = os.path.join(base_target_dir, folder_name)
    os.makedirs(full_path, exist_ok=True)
    session_paths[session_name] = full_path  # 保存路径引用以备后用

# 遍历CSV文件并筛选
for filename in os.listdir(source_dir):
    if filename.lower().endswith(".csv"):
        file_path = os.path.join(source_dir, filename)
        try:
            df = pd.read_csv(file_path)

            # 检查必须的列
            if 'DATE' not in df.columns:
                print(f"[警告] 缺少 'DATE' 列: {filename}")
                continue
            if 'TITLE' not in df.columns:
                print(f"[警告] 缺少 'TITLE' 列: {filename}")
                continue

            # 筛选 TITLE 含 Speech
            title = str(df['TITLE'].iloc[0])
            if "Speech" not in title:
                continue

            # 日期解析
            date_str = str(df['DATE'].iloc[0]).strip()
            try:
                speech_date = datetime.strptime(date_str, "%B %d, %Y")
            except ValueError:
                print(f"[警告] 无法解析日期格式: {filename} -> '{date_str}'")
                continue

            # 判断 Session 并复制文件
            for session_name, (start_date, end_date) in sessions.items():
                if start_date <= speech_date <= end_date:
                    target_folder = session_paths[session_name]
                    shutil.copy(file_path, os.path.join(target_folder, filename))
                    break

        except Exception as e:
            print(f"[错误] 处理文件失败: {filename} -> {e}")


# Process2: clean csv

In [ ]:
import os
import pandas as pd
import re
from shutil import copyfile

# 原始路径和目标路径
raw_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\csv_raw"
cleaned_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\cleaned_csv"

# 创建 cleaned 基础目录（如不存在）
os.makedirs(cleaned_base, exist_ok=True)

# 用于记录异常文件
missing_columns_files = []
no_trump_files = []

# 遍历所有 session 文件夹
for session_folder in os.listdir(raw_base):
    session_path = os.path.join(raw_base, session_folder)
    if not os.path.isdir(session_path):
        continue

    # 创建对应的 cleaned 子目录
    cleaned_session_path = os.path.join(cleaned_base, session_folder)
    os.makedirs(cleaned_session_path, exist_ok=True)

    # 遍历 CSV 文件
    for filename in os.listdir(session_path):
        if not filename.lower().endswith('.csv'):
            continue

        file_path = os.path.join(session_path, filename)
        try:
            df = pd.read_csv(file_path)

            # 检查必要列
            if 'TEXT' not in df.columns or 'SPEAKER' not in df.columns:
                print(f"[警告] 缺少 TEXT 或 SPEAKER 列: {filename}")
                missing_columns_files.append(filename)
                continue

            # 过滤出 Donald Trump 的发言
            df_trump = df[df['SPEAKER'].astype(str).str.strip() == 'Donald Trump']
            if df_trump.empty:
                print(f"[警告] 无 Donald Trump 发言: {filename}")
                no_trump_files.append(filename)
                continue

            # 清洗 TEXT 列：移除方括号中的内容
            df_trump = df_trump.copy()  # 明确创建副本
            df_trump.loc[:, 'TEXT'] = df_trump['TEXT'].astype(str).apply(lambda x: re.sub(r"\[.*?\]", "", x).strip())


            # 保存清洗后的文件
            cleaned_file_path = os.path.join(cleaned_session_path, filename)
            df_trump.to_csv(cleaned_file_path, index=False)

        except Exception as e:
            print(f"[错误] 无法处理文件: {filename} -> {e}")

# 输出警告信息
print("\n=== 以下文件缺少必要列（TEXT 或 SPEAKER） ===")
for f in missing_columns_files:
    print(f"- {f}")

print("\n=== 以下文件不含 Donald Trump 发言 ===")
for f in no_trump_files:
    print(f"- {f}")


In [ ]:
import os
import pandas as pd
import re

# 输入路径
raw_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\csv_raw"
# 输出路径
cleaned_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\cleaned_csv"
os.makedirs(cleaned_base, exist_ok=True)

# 异常文件日志
missing_columns_files = []
no_trump_files = []

# 🔁 尝试 UTF-8 和 UTF-8 with BOM 读取文件
def read_csv_strict_utf(filepath):
    try:
        return pd.read_csv(filepath, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(filepath, encoding='utf-8-sig')

# 🔁 遍历每个 Session 子文件夹
for session_folder in os.listdir(raw_base):
    session_path = os.path.join(raw_base, session_folder)
    if not os.path.isdir(session_path):
        continue

    # 创建目标 cleaned 子文件夹
    cleaned_session_path = os.path.join(cleaned_base, session_folder)
    os.makedirs(cleaned_session_path, exist_ok=True)

    # 遍历所有 .csv 文件
    for filename in os.listdir(session_path):
        if not filename.lower().endswith('.csv'):
            continue

        file_path = os.path.join(session_path, filename)
        try:
            # ⬇ 读取 CSV 文件（UTF-8）
            df = read_csv_strict_utf(file_path)

            # 🛑 检查是否有 TEXT 和 SPEAKER 列
            if 'TEXT' not in df.columns or 'SPEAKER' not in df.columns:
                print(f"[警告] 缺少 TEXT 或 SPEAKER 列: {filename}")
                missing_columns_files.append(filename)
                continue

            # 🧹 仅保留 Donald Trump 发言行
            df_trump = df[df['SPEAKER'].astype(str).str.strip() == 'Donald Trump'].copy()
            if df_trump.empty:
                print(f"[警告] 无 Donald Trump 发言: {filename}")
                no_trump_files.append(filename)
                continue

            # 🧼 清理 TEXT 中的 [噪音]
            df_trump.loc[:, 'TEXT'] = df_trump['TEXT'].astype(str).apply(
                lambda x: re.sub(r"\[.*?\]", "", x).strip()
            )

            # 💾 生成保存路径，添加 _clean 后缀
            base_name = os.path.splitext(filename)[0]
            cleaned_filename = base_name + "_clean.csv"
            cleaned_file_path = os.path.join(cleaned_session_path, cleaned_filename)

            # ✅ 保存为 UTF-8
            df_trump.to_csv(cleaned_file_path, index=False, encoding='utf-8')

        except Exception as e:
            print(f"[错误] 无法处理文件: {filename} -> {e}")

# ✅ 输出总结日志
print("\n=== ⚠️ 以下文件缺少 TEXT 或 SPEAKER 列 ===")
for f in missing_columns_files:
    print(f"- {f}")

print("\n=== ⚠️ 以下文件中无 Donald Trump 发言 ===")
for f in no_trump_files:
    print(f"- {f}")


In [ ]:
import os
import pandas as pd
import re

# 输入路径
raw_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\csv_raw"
# 输出路径
cleaned_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\cleaned_csv"
os.makedirs(cleaned_base, exist_ok=True)

# 异常文件日志
missing_columns_files = []
no_trump_files = []

# 读取 CSV（仅限 UTF-8 和 UTF-8-BOM）
def read_csv_strict_utf(filepath):
    try:
        return pd.read_csv(filepath, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(filepath, encoding='utf-8-sig')

# 非英文字符清除函数
def clean_english_only(text):
    text = re.sub(r"\[.*?\]", "", text)  # 去掉方括号内容
    text = re.sub(r"[^a-zA-Z0-9.,;:'\"!?()\- \n]", "", text)  # 去掉非英文字符
    text = re.sub(r"\s+", " ", text)  # 合并多余空格
    return text.strip()

# 遍历每个 Session 子文件夹
for session_folder in os.listdir(raw_base):
    session_path = os.path.join(raw_base, session_folder)
    if not os.path.isdir(session_path):
        continue

    cleaned_session_path = os.path.join(cleaned_base, session_folder)
    os.makedirs(cleaned_session_path, exist_ok=True)

    for filename in os.listdir(session_path):
        if not filename.lower().endswith('.csv'):
            continue

        file_path = os.path.join(session_path, filename)
        try:
            df = read_csv_strict_utf(file_path)

            if 'TEXT' not in df.columns or 'SPEAKER' not in df.columns:
                print(f"[警告] 缺少 TEXT 或 SPEAKER 列: {filename}")
                missing_columns_files.append(filename)
                continue

            df_trump = df[df['SPEAKER'].astype(str).str.strip() == 'Donald Trump'].copy()
            if df_trump.empty:
                print(f"[警告] 无 Donald Trump 发言: {filename}")
                no_trump_files.append(filename)
                continue

            df_trump['TEXT'] = df_trump['TEXT'].astype(str).apply(clean_english_only)

            base_name = os.path.splitext(filename)[0]
            cleaned_filename = base_name + "_clean.csv"
            cleaned_file_path = os.path.join(cleaned_session_path, cleaned_filename)
            df_trump.to_csv(cleaned_file_path, index=False, encoding='utf-8')

        except Exception as e:
            print(f"[错误] 无法处理文件: {filename} -> {e}")

# 输出总结
print("\n=== ⚠️ 缺失 TEXT 或 SPEAKER 的文件 ===")
for f in missing_columns_files:
    print(f"- {f}")

print("\n=== ⚠️ 无 Donald Trump 发言的文件 ===")
for f in no_trump_files:
    print(f"- {f}")


# Process3: get txt

In [ ]:
import os
import pandas as pd
import re

# 输入路径：清洗后的 CSV 文件夹
cleaned_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\cleaned_csv"
# 输出路径：TXT 文件夹
txt_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\txt"

# 确保基础 txt 文件夹存在
os.makedirs(txt_base, exist_ok=True)

# 遍历所有 session 文件夹
for session_folder in os.listdir(cleaned_base):
    session_path = os.path.join(cleaned_base, session_folder)
    if not os.path.isdir(session_path):
        continue

    # 创建对应的 txt 子目录
    txt_session_path = os.path.join(txt_base, session_folder)
    os.makedirs(txt_session_path, exist_ok=True)

    # 遍历 CSV 文件
    for filename in os.listdir(session_path):
        if not filename.lower().endswith('.csv'):
            continue

        file_path = os.path.join(session_path, filename)
        try:
            df = pd.read_csv(file_path)

            if 'TEXT' not in df.columns:
                print(f"[警告] 缺少 TEXT 列: {filename}")
                continue

            # 合并文本
            text_list = df['TEXT'].dropna().astype(str).tolist()
            combined_text = ' '.join(text_list)

            # 清洁拼写：去除多余空格、控制字符等
            cleaned_text = re.sub(r"\s+", " ", combined_text).strip()

            # 保存为 txt 文件
            txt_filename = os.path.splitext(filename)[0] + "_text.txt"
            txt_path = os.path.join(txt_session_path, txt_filename)

            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(cleaned_text)

        except Exception as e:
            print(f"[错误] 无法处理文件: {filename} -> {e}")


# Process4 (Damn)

In [ ]:
import os
import shutil
import pandas as pd
import re
from collections import defaultdict

# 原始 txt 路径
txt_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\txt"
# 目标路径
txt_short_base = r"C:\Users\numinose\Desktop\Trump in factbase\Trump Speech Study\txt_short"
os.makedirs(txt_short_base, exist_ok=True)

# 存储映射信息
mapping_records = []

# 用于记录重复短名计数
filename_tracker = defaultdict(int)

# 正则匹配日期
date_pattern = re.compile(r"_?-?([A-Z][a-z]+_\d{1,2}_\d{4})")

# 遍历 Session 文件夹
for session_folder in os.listdir(txt_base):
    session_path = os.path.join(txt_base, session_folder)
    if not os.path.isdir(session_path):
        continue

    # 创建短名 Session 子文件夹
    short_session_path = os.path.join(txt_short_base, session_folder)
    os.makedirs(short_session_path, exist_ok=True)

    for filename in os.listdir(session_path):
        if not filename.endswith('.txt'):
            continue

        original_path = os.path.join(session_path, filename)

        # 去掉 "Speech_" 和 "_clean_text.txt"
        base_name = filename.replace("Speech_", "").replace("_clean_text.txt", "")

        # 去掉尾部日期部分
        base_name = re.sub(date_pattern, "", base_name).strip("_-")

        # 生成目标名
        short_name_base = base_name + ".txt"
        short_name_final = short_name_base

        # 检查是否重复，若重复则编号
        key = os.path.join(session_folder, short_name_base)
        if filename_tracker[key] > 0:
            suffix = f"_{filename_tracker[key]}"
            short_name_final = base_name + suffix + ".txt"
        filename_tracker[key] += 1

        # 拷贝并重命名
        dest_path = os.path.join(short_session_path, short_name_final)
        shutil.copyfile(original_path, dest_path)

        # 添加到映射表
        mapping_records.append({
            "session": session_folder,
            "long_filename": filename,
            "short_filename": short_name_final
        })

# 保存映射表
mapping_df = pd.DataFrame(mapping_records)
mapping_path = os.path.join(txt_short_base, "filename_mapping.csv")
mapping_df.to_csv(mapping_path, index=False, encoding='utf-8')

print(f"\n✅ 文件名缩短完成，映射表保存于：{mapping_path}")
